# 5TFT-ELU-UnitNorm

In [ ]:
!pip install optuna pytorch-forecasting lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.4/848.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 8.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import random
import os
import json
import warnings
import logging
from collections import Counter

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE
from pytorch_forecasting.models.temporal_fusion_transformer import sub_modules
from pytorch_forecasting.models.temporal_fusion_transformer.sub_modules import (
    ResampleNorm, GateAddNorm
)
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Isi dengan nama file yang ada di dataset Kaggle-mu
file_path = "4TFT-ELU-UnitNorm.db"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "wahyusatriowidodo/tftprune",
    file_path,
    # Karena ini file .db, kamu wajib memberikan query SQL
    sql_query="SELECT * FROM trials"
)

print(df.head())


   trial_id  number  study_id     state              datetime_start  \
0         1       0         1  COMPLETE  2026-05-18 01:16:36.679007   
1         2       1         1  COMPLETE  2026-05-18 01:24:40.139992   
2         3       2         1  COMPLETE  2026-05-18 01:54:13.614009   
3         4       3         1  COMPLETE  2026-05-18 02:12:53.462271   
4         5       4         1  COMPLETE  2026-05-18 02:28:08.700072   

            datetime_complete  
0  2026-05-18 01:24:40.093899  
1  2026-05-18 01:54:13.571178  
2  2026-05-18 02:12:53.432956  
3  2026-05-18 02:28:08.670571  
4  2026-05-18 02:35:21.387476  


In [ ]:
# ===============================
# 0️⃣ CONFIGURATION
# ===============================
import os
import shutil # Tambahkan ini untuk fungsi copy file
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

ACTIVATION_NAME  = "ELU"
INIT_NAME        = "UnitNorm"
GRN_CLASS_NAME   = "GRNELU_UnitNorm"
EXPERIMENT_NAME  = f"{ACTIVATION_NAME}_{INIT_NAME}"          # "GELU_Kaiming"
STUDY_NAME       = f"tft_walkforward_10fold_{EXPERIMENT_NAME}"
RESULTS_FILENAME = f"results_{EXPERIMENT_NAME}_10fold.json"
BASE_DIR = f"/kaggle/working/{EXPERIMENT_NAME}"
os.makedirs(BASE_DIR, exist_ok=True)
STORAGE          = f"sqlite:///{BASE_DIR}/4TFT-ELU-UnitNorm.db"
WORKING_DB_PATH  = f"{BASE_DIR}/4TFT-ELU-UnitNorm.db"
UPLOADED_DB_DIR  = "/kaggle/input/datasets/wahyusatriowidodo/tftprune"
UPLOADED_DB_PATH = f"{UPLOADED_DB_DIR}/4TFT-ELU-UnitNorm.db"

print("="*80)
print("🔍 STATUS DATABASE OPTUNA:")
if os.path.exists(WORKING_DB_PATH):
    print("   ✅ Melanjutkan dari sesi yang sedang berjalan (Working Directory).")
elif os.path.exists(UPLOADED_DB_DIR) and os.path.exists(UPLOADED_DB_PATH):
    print(f"   📥 Database backup ditemukan di Input Directory!")
    print("      Menyalin ke folder working...")
    shutil.copy(UPLOADED_DB_PATH, WORKING_DB_PATH)
    print("   ✅ Copy berhasil! Siap melanjutkan trials yang tersisa.")
else:
    print("   🆕 Tidak ada database backup yang ditemukan.")
    print("      Memulai eksperimen dari AWAL (Trial 0).")
print("="*80)

OPTIONAL_FEATURES = [
    "SMA_10", "WMA_10", "MACD", "Momentum",
    "Stoch_K", "Stoch_D", "RSI", "Williams_R", "CCI", "MFI"
]

ENCODER_LENGTH                       = 30
EVAL_WINDOW                          = 21   # 21 prediksi 1-step per fold → RMSE stabil
MAX_PREDICTION_LENGTH                = 1
EARLY_STOPPING_PATIENCE_OPTIMIZATION = 5
EARLY_STOPPING_PATIENCE_FINAL        = 10
N_STARTUP_TRIALS                     = 24
N_WARMUP_STEPS                       = 3
N_FOLDS                              = 10
N_TRIALS                             = 240

🔍 STATUS DATABASE OPTUNA:
   📥 Database backup ditemukan di Input Directory!
      Menyalin ke folder working...
   ✅ Copy berhasil! Siap melanjutkan trials yang tersisa.


In [ ]:
if os.path.exists(WORKING_DB_PATH):
    try:
        storage_check = optuna.storages.RDBStorage(url=STORAGE)
        summaries = optuna.get_all_study_summaries(storage=storage_check)
        print("\n🔍 DAFTAR STUDY DI DALAM DATABASE INI:")
        for s in summaries:
            print(f"   - Nama: '{s.study_name}' | Total Trials: {s.n_trials}")
    except Exception as e:
        print(f"⚠️ Error membaca database: {e}")
print("="*60)


🔍 DAFTAR STUDY DI DALAM DATABASE INI:
   - Nama: 'tft_walkforward_10fold_ELU_UnitNorm' | Total Trials: 229


In [ ]:
# ===============================
# 🔧 CUSTOM GRN CLASS
# Aktivasi : ELU
# Init     : UnitNorm
#            Identik dengan implementasi resmi pytorch-forecasting
# ===============================
class GRNELU_UnitNorm(nn.Module):
    """
    GRN dengan aktivasi ELU dan inisialisasi UnitNorm.

    Init Kaiming Normal dipilih untuk fc1/fc2 karena dirancang untuk
    aktivasi ReLU-like (He et al., 2015), dan secara empiris bekerja
    baik untuk GELU yang memiliki karakteristik serupa (smooth near-zero).
    Init Xavier Uniform digunakan untuk layer context sesuai implementasi
    resmi pytorch-forecasting (Glorot & Bengio, 2010).

    Referensi:
    - He et al. (2015). Delving deep into rectifiers.
    - Glorot & Bengio (2010). Understanding the difficulty of training DNNs.
    - pytorch-forecasting source: sub_modules.GatedResidualNetwork
    """
    ACTIVATION_NAME = "ELU"
    INIT_NAME       = "UnitNorm"

    def __init__(self, input_size, hidden_size, output_size,
                 dropout=0.1, context_size=None, residual=False):
        super().__init__()
        self.input_size   = input_size
        self.output_size  = output_size
        self.context_size = context_size
        self.hidden_size  = hidden_size
        self.dropout      = dropout
        self.residual     = residual

        if self.input_size != self.output_size and not self.residual:
            residual_size = self.input_size
        else:
            residual_size = self.output_size
        if self.output_size != residual_size:
            self.resample_norm = ResampleNorm(residual_size, self.output_size)

        self.fc1        = nn.Linear(self.input_size, self.hidden_size)
        self.activation = nn.ELU()                                     # ← ELU
        if self.context_size is not None:
            self.context = nn.Linear(self.context_size, self.hidden_size, bias=False)
        self.fc2       = nn.Linear(self.hidden_size, self.hidden_size)
        self.gate_norm = GateAddNorm(
            input_size=self.hidden_size, skip_size=self.output_size,
            hidden_size=self.output_size, dropout=self.dropout, trainable_add=False,
        )
        self.init_weights()

    def init_weights(self):
        """
        Unit Norm Rows — setiap baris weight matrix memiliki L2 norm = 1.
        Referensi: Hendrycks & Gimpel (2016); Mishkin & Matas (2016);
        Saxe et al. (2014).
        """
        for name, p in self.named_parameters():
            if "bias" in name:
                torch.nn.init.zeros_(p)
            elif "fc1" in name or "fc2" in name or "context" in name:
                torch.nn.init.normal_(p)
                with torch.no_grad():
                    p.data = F.normalize(p.data, dim=1)

    def forward(self, x, context=None, residual=None):
        if residual is None:
            residual = x
        if self.input_size != self.output_size and not self.residual:
            residual = self.resample_norm(residual)
        x = self.fc1(x)
        if context is not None:
            context = self.context(context)
            x = x + context
        x = self.activation(x)
        x = self.fc2(x)
        x = self.gate_norm(x, residual)
        return x


# ===============================
# ✅ PATCH & VALIDASI GRN
# ===============================
def patch_and_validate_grn(grn_class):
    assert grn_class.ACTIVATION_NAME in ["ELU", "RELU", "GELU"]
    assert grn_class.INIT_NAME in ["Kaiming", "UnitNorm"]
    sub_modules.GatedResidualNetwork = grn_class
    patched = sub_modules.GatedResidualNetwork
    assert patched.__name__ == grn_class.__name__
    dummy           = grn_class(input_size=8, hidden_size=8, output_size=8, dropout=0.1)
    activation_type = type(dummy.activation).__name__
    activation_map  = {"ELU": "ELU", "RELU": "ReLU", "GELU": "GELU"}
    assert activation_type == activation_map[grn_class.ACTIVATION_NAME], \
        f"❌ Aktivasi salah! Expected {activation_map[grn_class.ACTIVATION_NAME]}, got {activation_type}"
    print(f"✅ Patch  : {patched.__name__}")
    print(f"✅ Aktivasi: {activation_type}")
    print(f"✅ Init    : {grn_class.INIT_NAME}")
    return True

print("\n" + "="*80)
print("PATCHING GRN CLASS → GRNELU_UnitNorm")
print("="*80)
patch_and_validate_grn(GRNELU_UnitNorm)


PATCHING GRN CLASS → GRNELU_UnitNorm
✅ Patch  : GRNELU_UnitNorm
✅ Aktivasi: ELU
✅ Init    : UnitNorm


True

In [ ]:
import gdown
files = {
    'df.csv': '11-HR0CLqEn9dDwaJRVc6PqM8LdEYktf5'
}

for name, file_id in files.items():
    print(f'Downloading {file_id} to {name}')
    gdown.download(id=file_id, output=name, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=11-HR0CLqEn9dDwaJRVc6PqM8LdEYktf5
To: /kaggle/working/df.csv
100%|██████████| 459k/459k [00:00<00:00, 86.3MB/s]


In [ ]:
# ===============================
# 1️⃣ DATA LOADING & PREPROCESSING
# ===============================
print("="*80 + f"\nEKSPERIMEN: {EXPERIMENT_NAME}\n" + "="*80)

df = pd.read_csv('df.csv', encoding='latin-1')
#df.drop(columns=['volume', 'pos_mean', 'neg_mean', 'neu_mean'], inplace=True)
df["tanggal"] = pd.to_datetime(df["tanggal"])
df = df.sort_values("tanggal").reset_index(drop=True)
df = df.replace([np.inf, -np.inf], np.nan)
for col in df.columns:
    if "sentiment" in col:
        df[col] = df[col].fillna(0.0)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().fillna(0.0)
assert not df.isna().any().any(), "❌ Masih ada NaN setelah preprocessing"

df["sentiment_lag1"] = df["vader"].shift(1)
df = df.iloc[1:].reset_index(drop=True)
assert not df["sentiment_lag1"].isna().any()

df["series_id"] = 0
df["close"]     = df["close"].astype(float)
df["time_idx"]  = np.arange(len(df))

print(f"📊 Total samples : {len(df)}")
print(f"📅 Date range    : {df['tanggal'].min().date()} → {df['tanggal'].max().date()}")
print(f"🔧 Aktivasi      : {ACTIVATION_NAME}")
print(f"🔧 Init          : {INIT_NAME}")

EKSPERIMEN: ELU_UnitNorm
📊 Total samples : 2434
📅 Date range    : 2016-01-06 → 2025-12-30
🔧 Aktivasi      : ELU
🔧 Init          : UnitNorm


In [ ]:
# ===============================
# 2️⃣ METRICS
# ===============================
def calculate_metrics(actuals, predictions, train_actuals=None):
    """
    Menghitung metrik evaluasi prediksi time series.

    Parameters
    ----------
    actuals       : array actual (val atau test)
    predictions   : array prediksi model
    train_actuals : array harga closing training — diperlukan untuk MASE.
                    Jika None, MASE tidak dihitung (dikembalikan sebagai NaN).

    Metrik:
    - RMSE  : Root Mean Squared Error
    - MAPE  : Mean Absolute Percentage Error (%)
    - sMAPE : Symmetric MAPE (%)
    - MASE  : Mean Absolute Scaled Error
              = MAE_model / MAE_naive
              MAE_naive = mean|y_t - y_{t-1}| pada data training
              MASE < 1 → model lebih baik dari naive forecast
              (Hyndman & Koehler, 2006)
    - DA    : Directional Accuracy (%)
              = % prediksi yang arahnya (naik/turun) benar
    """
    eps = 1e-8
    actuals     = np.array(actuals)
    predictions = np.array(predictions)

    rmse  = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape  = np.mean(np.abs((actuals - predictions) / (actuals + eps))) * 100
    smape = 100 * np.mean(2 * np.abs(predictions - actuals) /
                          (np.abs(actuals) + np.abs(predictions) + eps))

    # MASE — Mean Absolute Scaled Error (Hyndman & Koehler, 2006)
    if train_actuals is not None and len(train_actuals) > 1:
        train_actuals  = np.array(train_actuals)
        mae_naive      = np.mean(np.abs(np.diff(train_actuals)))   # naive 1-step
        mae_model      = np.mean(np.abs(actuals - predictions))
        mase           = mae_model / (mae_naive + eps)
    else:
        mase = float("nan")

    # Directional Accuracy
    if len(actuals) >= 2 and len(predictions) >= 2:
        actual_dir = np.diff(actuals)
        pred_dir   = np.diff(predictions)
        # Arah sama jika keduanya naik (>0), keduanya turun (<0),
        # atau keduanya tidak berubah (==0)
        # Gunakan np.sign untuk konsistensi, skip jika actual_dir == 0
        # (tidak ada arah yang bisa dinilai benar/salah)
        mask = actual_dir != 0   # skip titik yang actual-nya flat
        if mask.sum() > 0:
            correct = (np.sign(actual_dir[mask]) == np.sign(pred_dir[mask]))
            da = float(np.mean(correct) * 100)
        else:
            da = float("nan")
    else:
        da = float("nan")

    return {"rmse": rmse, "mape": mape, "smape": smape, "mase": mase, "da": da}

In [ ]:
# ===============================
# 3️⃣ WALK-FORWARD SPLIT
# ===============================
print("\n" + "="*80)
print(f"WALK-FORWARD VALIDATION — {N_FOLDS} FOLDS × {EVAL_WINDOW} PREDIKSI")
print("="*80)

total_samples  = len(df)
min_train_size = ENCODER_LENGTH + MAX_PREDICTION_LENGTH + 10

# val/test size = ENCODER_LENGTH + EVAL_WINDOW
# = 30 + 21 = 51 → 21 prediksi 1-step per fold
# Sebelumnya 31 (hanya 1 prediksi/fold → RMSE tidak stabil)
test_size          = ENCODER_LENGTH + EVAL_WINDOW   # 51
val_size           = test_size                       # 51
initial_train_size = total_samples - (N_FOLDS * (val_size + test_size))

assert initial_train_size >= min_train_size, \
    f"❌ Data tidak cukup! initial_train={initial_train_size} < min={min_train_size}"

print(f"   Total samples          : {total_samples}")
print(f"   Encoder length         : {ENCODER_LENGTH}")
print(f"   Eval window            : {EVAL_WINDOW} prediksi 1-step per fold")
print(f"   Val/Test size per fold : {val_size} ({ENCODER_LENGTH} encoder + {EVAL_WINDOW} eval)")
print(f"   Initial train size     : {initial_train_size}")


WALK-FORWARD VALIDATION — 10 FOLDS × 21 PREDIKSI
   Total samples          : 2434
   Encoder length         : 30
   Eval window            : 21 prediksi 1-step per fold
   Val/Test size per fold : 51 (30 encoder + 21 eval)
   Initial train size     : 1414


In [ ]:
# ===============================
# 4️⃣ CREATE FOLD SPLITS
# ===============================
def create_walk_forward_splits(df, n_folds, initial_train_size, val_size, test_size):
    splits = []
    for fold in range(n_folds):
        train_end = initial_train_size + fold * (val_size + test_size)
        val_end   = train_end + val_size
        test_end  = val_end  + test_size
        if test_end > len(df): break
        tr = df.iloc[:train_end].copy().reset_index(drop=True)
        vl = df.iloc[train_end:val_end].copy().reset_index(drop=True)
        te = df.iloc[val_end:test_end].copy().reset_index(drop=True)
        splits.append((tr, vl, te))
        print(f"  Fold {fold+1:2d}: Train {len(tr):4d} | Val {len(vl)} ({EVAL_WINDOW} pred) | "
              f"Test {len(te)} ({EVAL_WINDOW} pred) | "
              f"{tr['tanggal'].min().date()} → {te['tanggal'].max().date()}")
    return splits

fold_splits = create_walk_forward_splits(df, N_FOLDS, initial_train_size, val_size, test_size)

for tr, vl, te in fold_splits:
    assert tr["time_idx"].max() + 1 == vl["time_idx"].min()
    assert vl["time_idx"].max() + 1 == te["time_idx"].min()
    assert np.isclose(tr.iloc[-1]["vader"], vl.iloc[0]["sentiment_lag1"], rtol=1e-5)
    assert np.isclose(vl.iloc[-1]["vader"], te.iloc[0]["sentiment_lag1"], rtol=1e-5)
print(f"✅ {len(fold_splits)} fold valid")

  Fold  1: Train 1414 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2022-02-25
  Fold  2: Train 1516 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2022-08-04
  Fold  3: Train 1618 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2022-12-27
  Fold  4: Train 1720 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2023-06-06
  Fold  5: Train 1822 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2023-11-03
  Fold  6: Train 1924 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2024-04-16
  Fold  7: Train 2026 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2024-09-18
  Fold  8: Train 2128 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2025-02-19
  Fold  9: Train 2230 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2025-08-04
  Fold 10: Train 2332 | Val 51 (21 pred) | Test 51 (21 pred) | 2016-01-06 → 2025-12-30
✅ 10 fold valid


In [ ]:
# ===============================
# 5️⃣ TRAINING FUNCTION
# ===============================
def train_eval_tft(train_df, val_df, features, cfg, verbose=False, fold_idx=None):
    try:
        training = TimeSeriesDataSet(
            train_df, time_idx="time_idx", target="close", group_ids=["series_id"],
            max_encoder_length=cfg["encoder_len"],
            max_prediction_length=MAX_PREDICTION_LENGTH,
            time_varying_known_reals=["time_idx"],
            time_varying_unknown_reals=features,
            add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        )
        validation = TimeSeriesDataSet.from_dataset(
            training, val_df, predict=True, stop_randomization=True
        )
        nw = 2 if len(train_df) > 500 else 0
        tl = training.to_dataloader(
            train=True, batch_size=cfg["batch"], num_workers=nw,
            pin_memory=torch.cuda.is_available(), persistent_workers=(nw > 0))
        vl = validation.to_dataloader(
            train=False, batch_size=cfg["batch"], num_workers=nw,
            pin_memory=torch.cuda.is_available(), persistent_workers=(nw > 0))

        class TFTNoWarn(TemporalFusionTransformer):
            def __init__(self, *a, **kw):
                super().__init__(*a, **kw)
                self.save_hyperparameters(ignore=["loss", "logging_metrics"])

        model = TFTNoWarn.from_dataset(
            training, hidden_size=cfg["hidden"], dropout=cfg["dropout"],
            attention_head_size=cfg["heads"], learning_rate=cfg["lr"], loss=RMSE(),
        )
        trainer = Trainer(
            max_epochs=50,
            accelerator="gpu" if torch.cuda.is_available() else "cpu",
            precision="32", gradient_clip_val=cfg["clipnorm"],
            enable_checkpointing=False, logger=False,
            enable_progress_bar=verbose, enable_model_summary=False,
            profiler=None, deterministic=False, log_every_n_steps=1,
            callbacks=[EarlyStopping(monitor="val_loss",
                                     patience=EARLY_STOPPING_PATIENCE_OPTIMIZATION,
                                     mode="min")],
        )
        trainer.fit(model, tl, vl)

        val_loss   = trainer.callback_metrics["val_loss"].item()
        train_loss = trainer.callback_metrics.get("train_loss_epoch")
        epochs_run = trainer.current_epoch
        gap = gap_pct = None
        status = "train_loss tidak tersedia"

        if train_loss is not None:
            train_loss = train_loss.item()
            gap        = val_loss - train_loss
            gap_pct    = (gap / (train_loss + 1e-8)) * 100
            # Logika status berdasarkan gap_pct:
            # gap > 20%  → val jauh lebih buruk dari train  → OVERFIT
            # gap < -50% → val jauh lebih baik dari train   → anomali (val kecil/mudah)
            # gap < 0%   → val sedikit lebih baik           → normal (dropout effect)
            # 0–20%      → val sedikit lebih buruk          → fit baik
            if   gap_pct > 20:  status = "⚠️ OVERFIT"
            elif gap_pct < -50: status = "⚠️ VAL SANGAT RENDAH (periode mudah/noise)"
            elif gap_pct < 0:   status = "✅ NORMAL (generalisasi baik)"
            else:               status = "✅ FIT BAIK"
            label = f"Fold {fold_idx}" if fold_idx else "Fold ?"
            print(f"    {label} | train={train_loss:.4f}  val={val_loss:.4f}  "
                  f"gap={gap:+.4f} ({gap_pct:+.1f}%)  ep={epochs_run}  {status}")
        else:
            print(f"    Fold {fold_idx} | val={val_loss:.4f}  ep={epochs_run}")

        return val_loss, {
            "train_loss": train_loss, "val_loss": val_loss,
            "gap": gap, "gap_pct": gap_pct,
            "status": status, "epochs": epochs_run,
        }
    except Exception as e:
        print(f"❌ Error fold {fold_idx}: {e}")
        return float('inf'), {}

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)
import logging
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)


# Pruning

In [ ]:

# ===============================
# 6️⃣ OPTUNA OBJECTIVE
# ===============================
def objective(trial):
    hidden   = trial.suggest_categorical("hidden",   [10, 20, 40, 80, 160, 240, 320])
    dropout  = trial.suggest_categorical("dropout",  [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9])
    batch    = trial.suggest_categorical("batch",    [64, 128, 256])
    lr       = trial.suggest_categorical("lr",       [1e-4, 1e-3, 1e-2])
    clipnorm = trial.suggest_categorical("clipnorm", [0.01, 1.0, 100.0])
    heads    = trial.suggest_categorical("heads",    [1, 4])

    trial.suggest_categorical("use_close",          [1])
    trial.suggest_categorical("use_sentiment_lag1", [1])
    for f in OPTIONAL_FEATURES:
        trial.suggest_categorical(f"use_{f}", [0, 1])

    cfg = {"hidden": hidden, "dropout": dropout, "batch": batch,
           "lr": lr, "clipnorm": clipnorm, "heads": heads, "encoder_len": ENCODER_LENGTH}
    selected = [k.replace("use_", "") for k, v in trial.params.items()
                if k.startswith("use_") and v == 1]

    for k, v in cfg.items(): trial.set_user_attr(k, v)
    trial.set_user_attr("n_features", len(selected))
    trial.set_user_attr("features",   selected)

    print(f"\n[Trial {trial.number}] {EXPERIMENT_NAME} | hidden={hidden}, "
          f"dropout={dropout}, batch={batch}, lr={lr}")
    print(f"  Features ({len(selected)}): {', '.join(selected[:5])}"
          f"{'...' if len(selected) > 5 else ''}")
    print(f"  {'─'*60}")

    rmses, gap_infos = [], []
    for fold_idx in range(len(fold_splits)):
        tr, vl, _ = fold_splits[fold_idx]
        rmse, gap_info = train_eval_tft(tr, vl, selected, cfg, fold_idx=fold_idx + 1)
        rmses.append(rmse)
        if gap_info: gap_infos.append(gap_info)

        running_avg = float(np.mean(rmses))
        # ─────────────────────────────────────────────────────────────────
        # PRUNING — MedianPruner (Akiba et al., 2019)
        # Setiap fold = 1 "step". MedianPruner dipilih karena:
        # (1) ASHA/Hyperband untuk learning curve epoch (Li et al., 2018),
        #     tidak cocok untuk fold-level evaluation
        # (2) WilcoxonPruner asumsikan independensi antar fold; dilanggar
        #     oleh walk-forward expanding window (train fold N ⊃ fold N-1)
        # (3) Running avg RMSE berkorelasi r≈0.98 dengan nilai akhir
        #     mulai fold ke-3 (Piotrek, 2019) → MedianPruner valid
        # n_warmup_steps=2: fold 1-2 sebagai warm-up, pruning aktif fold ke-3
        # ─────────────────────────────────────────────────────────────────
        trial.report(running_avg, step=fold_idx)
        if trial.should_prune():
            print(f"  ✂️  Trial {trial.number} dipruning setelah fold {fold_idx + 1} "
                  f"(running avg RMSE: {running_avg:.4f})")
            raise optuna.exceptions.TrialPruned()

    avg_rmse = float(np.mean(rmses))

    if gap_infos:
        overfit_cnt  = sum(1 for g in gap_infos if "OVERFIT"            in g.get("status", ""))
        anomali_cnt  = sum(1 for g in gap_infos if "VAL SANGAT RENDAH"  in g.get("status", ""))
        normal_cnt   = sum(1 for g in gap_infos if "NORMAL"             in g.get("status", ""))
        fit_cnt      = sum(1 for g in gap_infos if "FIT BAIK"           in g.get("status", ""))
        valid_gaps   = [g["gap_pct"] for g in gap_infos if g.get("gap_pct") is not None]
        avg_gap_pct  = float(np.mean(valid_gaps)) if valid_gaps else None
        avg_epochs   = float(np.mean([g["epochs"] for g in gap_infos]))

        trial.set_user_attr("avg_gap_pct",   avg_gap_pct)
        trial.set_user_attr("avg_epochs",    avg_epochs)
        trial.set_user_attr("overfit_folds", overfit_cnt)
        trial.set_user_attr("anomali_folds", anomali_cnt)
        trial.set_user_attr("normal_folds",  normal_cnt)
        trial.set_user_attr("fit_folds",     fit_cnt)

        print(f"\n  {'─'*60}")
        print(f"  📊 Trial {trial.number} [{EXPERIMENT_NAME}]: "
              f"Avg RMSE={avg_rmse:.4f} ± {np.std(rmses):.4f}")
        if avg_gap_pct is not None:
            print(f"     Avg Gap     : {avg_gap_pct:+.1f}%")
        print(f"     ✅ Normal/Fit: {normal_cnt + fit_cnt}/{len(gap_infos)}  "
              f"⚠️ Overfit: {overfit_cnt}/{len(gap_infos)}  "
              f"⚠️ Anomali: {anomali_cnt}/{len(gap_infos)}")

    return avg_rmse


In [ ]:

# ===============================
# 7️⃣ HYPERPARAMETER OPTIMIZATION
# ===============================
import sqlite3
from optuna.trial import TrialState

print("\n" + "="*80)
print(f"HYPERPARAMETER OPTIMIZATION — {EXPERIMENT_NAME} ({GRN_CLASS_NAME})")
print("="*80)

storage = optuna.storages.RDBStorage(url=STORAGE)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="minimize",
    sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS,
                       n_ei_candidates=24, multivariate=True),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=N_STARTUP_TRIALS,
        n_warmup_steps=N_WARMUP_STEPS,
        interval_steps=1,
    ),
    storage=storage,
    load_if_exists=True,
)

# Bersihkan RUNNING trials dari sesi yang terputus
db_path = STORAGE.replace("sqlite:///", "")
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()
cur.execute("SELECT COUNT(*) FROM trials WHERE state = 0")
n_run   = cur.fetchone()[0]
if n_run > 0:
    print(f"🧹 Membersihkan {n_run} RUNNING trial yang tertinggal")
    cur.execute("UPDATE trials SET state = 3 WHERE state = 0")
    conn.commit()
conn.close()

study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage,
    sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS, n_ei_candidates=24, multivariate=True),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=N_STARTUP_TRIALS, # Akan kembali jadi 24
        n_warmup_steps=N_WARMUP_STEPS,     # Akan kembali jadi 3
        interval_steps=1,
    )
)
state_counter = Counter(t.state for t in study.trials)
print("📊 Status trial saat ini:")
for k, v in state_counter.items():
    print(f"   {k}: {v}")

# Hitung semua trial yang sudah dievaluasi (baik yang selesai maupun yang dipotong/pruned)
n_finished = sum(1 for t in study.trials if t.state in [TrialState.COMPLETE, TrialState.PRUNED])
n_new      = max(0, N_TRIALS - n_finished)
if n_new == 0:
    print(f"✅ Optimasi sudah selesai ({N_TRIALS} trial)")
else:
    print(f"🚀 Menjalankan {n_new} trial baru (sudah {n_finished}/{N_TRIALS})")
    study.optimize(objective, n_trials=n_new, show_progress_bar=True,
                   catch=(KeyboardInterrupt,))

best = study.best_trial
print(f"\n🏆 Best Avg RMSE [{EXPERIMENT_NAME}]: {best.value:.4f}")

best_config = {
    "experiment":    EXPERIMENT_NAME,
    "activation":    ACTIVATION_NAME,
    "init":          INIT_NAME,
    "grn_class":     GRN_CLASS_NAME,
    "hidden":        best.user_attrs["hidden"],
    "dropout":       best.user_attrs["dropout"],
    "batch":         best.user_attrs["batch"],
    "lr":            best.user_attrs["lr"],
    "clipnorm":      best.user_attrs["clipnorm"],
    "heads":         best.user_attrs["heads"],
    "encoder_len":   ENCODER_LENGTH,
    "eval_window":   EVAL_WINDOW,
    "features":      best.user_attrs["features"],
    "best_avg_rmse": best.value,
    "avg_gap_pct":   best.user_attrs.get("avg_gap_pct"),
    "avg_epochs":    best.user_attrs.get("avg_epochs"),
    "overfit_folds": best.user_attrs.get("overfit_folds"),
    "anomali_folds": best.user_attrs.get("anomali_folds"),
    "normal_folds":  best.user_attrs.get("normal_folds"),
    "fit_folds":     best.user_attrs.get("fit_folds"),
}
with open(f"{BASE_DIR}/best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

display(study.trials_dataframe(attrs=("number", "state", "value", "params")))
optuna.visualization.plot_optimization_history(study).write_html(
    f"{BASE_DIR}/optimization_history.html")
FEATURE_PARAMS = [f"use_{f}" for f in OPTIONAL_FEATURES] + ["use_close", "use_sentiment_lag1"]
optuna.visualization.plot_param_importances(
    study, params=FEATURE_PARAMS
).write_html(f"{BASE_DIR}/param_importance.html")



HYPERPARAMETER OPTIMIZATION — ELU_UnitNorm (GRNELU_UnitNorm)


/tmp/ipykernel_16/2463671739.py:15: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS,
[I 2026-05-21 02:04:45,033] Using an existing study with name 'tft_walkforward_10fold_ELU_UnitNorm' instead of creating a new one.
/tmp/ipykernel_16/2463671739.py:41: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS, n_ei_candidates=24, multivariate=True),


📊 Status trial saat ini:
   TrialState.COMPLETE: 182
   TrialState.PRUNED: 42
   TrialState.RUNNING: 5
🚀 Menjalankan 16 trial baru (sudah 224/240)


  0%|          | 0/16 [00:00<?, ?it/s]


[Trial 229] ELU_UnitNorm | hidden=160, dropout=0.1, batch=64, lr=0.001
  Features (7): close, sentiment_lag1, SMA_10, WMA_10, MACD...
  ────────────────────────────────────────────────────────────


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


    Fold 3 | train=57.4212  val=27.1663  gap=-30.2550 (-52.7%)  ep=15  ⚠️ VAL SANGAT RENDAH (periode mudah/noise)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


    Fold 4 | train=81.1682  val=32.1238  gap=-49.0444 (-60.4%)  ep=6  ⚠️ VAL SANGAT RENDAH (periode mudah/noise)
  ✂️  Trial 244 dipruning setelah fold 4 (running avg RMSE: 68.7494)
[I 2026-05-21 06:11:51,298] Trial 244 pruned. 

🏆 Best Avg RMSE [ELU_UnitNorm]: 123.9783


,number,state,value,params_batch,params_clipnorm,params_dropout,params_heads,params_hidden,params_lr,params_use_CCI,...,params_use_MFI,params_use_Momentum,params_use_RSI,params_use_SMA_10,params_use_Stoch_D,params_use_Stoch_K,params_use_WMA_10,params_use_Williams_R,params_use_close,params_use_sentiment_lag1
0,0,COMPLETE,259.591638,256,0.01,0.5,4,20,0.0001,0,...,0,0,1,0,0,1,1,0,1,1
1,1,COMPLETE,131.354834,128,100.00,0.1,4,240,0.0001,0,...,0,0,0,0,0,0,1,1,1,1
2,2,COMPLETE,189.943713,64,100.00,0.5,1,160,0.0001,0,...,1,0,0,0,1,1,1,0,1,1
3,3,COMPLETE,169.646326,128,100.00,0.5,4,160,0.0100,0,...,0,0,0,1,0,1,0,0,1,1
4,4,COMPLETE,256.287463,256,0.01,0.7,1,40,0.0001,0,...,0,0,0,1,1,1,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,240,COMPLETE,136.297302,128,100.00,0.2,1,160,0.0010,0,...,0,0,1,0,0,0,1,0,1,1
241,241,COMPLETE,164.209973,128,100.00,0.4,1,160,0.0010,0,...,0,0,1,0,0,0,1,0,1,1
242,242,COMPLETE,161.591528,128,100.00,0.1,4,160,0.0100,1,...,1,1,0,0,0,0,1,1,1,1
243,243,COMPLETE,150.558374,128,100.00,0.1,1,240,0.0010,0,...,0,0,0,1,0,1,1,0,1,1


# No Pruning

In [ ]:
'''
# ===============================
# 6️⃣ OPTUNA OBJECTIVE
# ===============================
def objective(trial):
    hidden   = trial.suggest_categorical("hidden",   [10, 20, 40, 80, 160, 240, 320])
    dropout  = trial.suggest_categorical("dropout",  [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9])
    batch    = trial.suggest_categorical("batch",    [64, 128, 256])
    lr       = trial.suggest_categorical("lr",       [1e-4, 1e-3, 1e-2])
    clipnorm = trial.suggest_categorical("clipnorm", [0.01, 1.0, 100.0])
    heads    = trial.suggest_categorical("heads",    [1, 4])

    trial.suggest_categorical("use_close",          [1])
    trial.suggest_categorical("use_sentiment_lag1", [1])
    for f in OPTIONAL_FEATURES:
        trial.suggest_categorical(f"use_{f}", [0, 1])

    cfg = {"hidden": hidden, "dropout": dropout, "batch": batch,
           "lr": lr, "clipnorm": clipnorm, "heads": heads, "encoder_len": ENCODER_LENGTH}
    selected = [k.replace("use_", "") for k, v in trial.params.items()
                if k.startswith("use_") and v == 1]

    for k, v in cfg.items(): trial.set_user_attr(k, v)
    trial.set_user_attr("n_features", len(selected))
    trial.set_user_attr("features",   selected)

    print(f"\n[Trial {trial.number}] {EXPERIMENT_NAME} | hidden={hidden}, "
          f"dropout={dropout}, batch={batch}, lr={lr}")
    print(f"  Features ({len(selected)}): {', '.join(selected[:5])}"
          f"{'...' if len(selected) > 5 else ''}")
    print(f"  {'─'*60}")

    rmses, gap_infos = [], []
    for fold_idx in range(len(fold_splits)):
        tr, vl, _ = fold_splits[fold_idx]
        rmse, gap_info = train_eval_tft(tr, vl, selected, cfg, fold_idx=fold_idx + 1)
        rmses.append(rmse)
        if gap_info: gap_infos.append(gap_info)

        # Tanpa pruning — semua fold selalu dijalankan penuh
        running_avg = float(np.mean(rmses))
        print(f"  Fold {fold_idx + 1}/{len(fold_splits)} selesai | running avg RMSE: {running_avg:.4f}")

    avg_rmse = float(np.mean(rmses))

    if gap_infos:
        overfit_cnt  = sum(1 for g in gap_infos if "OVERFIT"            in g.get("status", ""))
        anomali_cnt  = sum(1 for g in gap_infos if "VAL SANGAT RENDAH"  in g.get("status", ""))
        normal_cnt   = sum(1 for g in gap_infos if "NORMAL"             in g.get("status", ""))
        fit_cnt      = sum(1 for g in gap_infos if "FIT BAIK"           in g.get("status", ""))
        valid_gaps   = [g["gap_pct"] for g in gap_infos if g.get("gap_pct") is not None]
        avg_gap_pct  = float(np.mean(valid_gaps)) if valid_gaps else None
        avg_epochs   = float(np.mean([g["epochs"] for g in gap_infos]))

        trial.set_user_attr("avg_gap_pct",   avg_gap_pct)
        trial.set_user_attr("avg_epochs",    avg_epochs)
        trial.set_user_attr("overfit_folds", overfit_cnt)
        trial.set_user_attr("anomali_folds", anomali_cnt)
        trial.set_user_attr("normal_folds",  normal_cnt)
        trial.set_user_attr("fit_folds",     fit_cnt)

        print(f"\n  {'─'*60}")
        print(f"  📊 Trial {trial.number} [{EXPERIMENT_NAME}]: "
              f"Avg RMSE={avg_rmse:.4f} ± {np.std(rmses):.4f}")
        if avg_gap_pct is not None:
            print(f"     Avg Gap     : {avg_gap_pct:+.1f}%")
        print(f"     ✅ Normal/Fit: {normal_cnt + fit_cnt}/{len(gap_infos)}  "
              f"⚠️ Overfit: {overfit_cnt}/{len(gap_infos)}  "
              f"⚠️ Anomali: {anomali_cnt}/{len(gap_infos)}")

    return avg_rmse
'''

'\n# ===============================\n# 6️⃣ OPTUNA OBJECTIVE\n# ===============================\ndef objective(trial):\n    hidden   = trial.suggest_categorical("hidden",   [10, 20, 40, 80, 160, 240, 320])\n    dropout  = trial.suggest_categorical("dropout",  [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9])\n    batch    = trial.suggest_categorical("batch",    [64, 128, 256])\n    lr       = trial.suggest_categorical("lr",       [1e-4, 1e-3, 1e-2])\n    clipnorm = trial.suggest_categorical("clipnorm", [0.01, 1.0, 100.0])\n    heads    = trial.suggest_categorical("heads",    [1, 4])\n\n    trial.suggest_categorical("use_close",          [1])\n    trial.suggest_categorical("use_sentiment_lag1", [1])\n    for f in OPTIONAL_FEATURES:\n        trial.suggest_categorical(f"use_{f}", [0, 1])\n\n    cfg = {"hidden": hidden, "dropout": dropout, "batch": batch,\n           "lr": lr, "clipnorm": clipnorm, "heads": heads, "encoder_len": ENCODER_LENGTH}\n    selected = [k.replace("use_", "") for k, v in trial.p

In [ ]:
'''
# ===============================
# 7️⃣ HYPERPARAMETER OPTIMIZATION
# ===============================
import sqlite3
from optuna.trial import TrialState

print("\n" + "="*80)
print(f"HYPERPARAMETER OPTIMIZATION — {EXPERIMENT_NAME} ({GRN_CLASS_NAME})")
print("="*80)

storage = optuna.storages.RDBStorage(url=STORAGE)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="minimize",
    sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS,
                       n_ei_candidates=24, multivariate=True),
    pruner=optuna.pruners.NopPruner(),   # tidak ada pruning — semua trial berjalan penuh
    storage=storage,
    load_if_exists=True,
)

# Bersihkan RUNNING trials dari sesi yang terputus
db_path = STORAGE.replace("sqlite:///", "")
conn    = sqlite3.connect(db_path)
cur     = conn.cursor()
cur.execute("SELECT COUNT(*) FROM trials WHERE state = 0")
n_run   = cur.fetchone()[0]
if n_run > 0:
    print(f"🧹 Membersihkan {n_run} RUNNING trial yang tertinggal")
    cur.execute("UPDATE trials SET state = 3 WHERE state = 0")
    conn.commit()
conn.close()

study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage,
    sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS,
                       n_ei_candidates=24, multivariate=True),
    pruner=optuna.pruners.NopPruner(),   # tidak ada pruning
)
state_counter = Counter(t.state for t in study.trials)
print("📊 Status trial saat ini:")
for k, v in state_counter.items():
    print(f"   {k}: {v}")

# Hitung trial yang sudah selesai penuh (tidak ada PRUNED karena NopPruner)
n_finished = sum(1 for t in study.trials if t.state == TrialState.COMPLETE)
n_new      = max(0, N_TRIALS - n_finished)
if n_new == 0:
    print(f"✅ Optimasi sudah selesai ({N_TRIALS} trial)")
else:
    print(f"🚀 Menjalankan {n_new} trial baru (sudah {n_finished}/{N_TRIALS})")
    study.optimize(objective, n_trials=n_new, show_progress_bar=True,
                   catch=(KeyboardInterrupt,))

best = study.best_trial
print(f"\n🏆 Best Avg RMSE [{EXPERIMENT_NAME}]: {best.value:.4f}")

best_config = {
    "experiment":    EXPERIMENT_NAME,
    "activation":    ACTIVATION_NAME,
    "init":          INIT_NAME,
    "grn_class":     GRN_CLASS_NAME,
    "hidden":        best.user_attrs["hidden"],
    "dropout":       best.user_attrs["dropout"],
    "batch":         best.user_attrs["batch"],
    "lr":            best.user_attrs["lr"],
    "clipnorm":      best.user_attrs["clipnorm"],
    "heads":         best.user_attrs["heads"],
    "encoder_len":   ENCODER_LENGTH,
    "eval_window":   EVAL_WINDOW,
    "features":      best.user_attrs["features"],
    "best_avg_rmse": best.value,
    "avg_gap_pct":   best.user_attrs.get("avg_gap_pct"),
    "avg_epochs":    best.user_attrs.get("avg_epochs"),
    "overfit_folds": best.user_attrs.get("overfit_folds"),
    "anomali_folds": best.user_attrs.get("anomali_folds"),
    "normal_folds":  best.user_attrs.get("normal_folds"),
    "fit_folds":     best.user_attrs.get("fit_folds"),
}
with open(f"{BASE_DIR}/best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

display(study.trials_dataframe(attrs=("number", "state", "value", "params")))
optuna.visualization.plot_optimization_history(study).write_html(
    f"{BASE_DIR}/optimization_history.html")
FEATURE_PARAMS = [f"use_{f}" for f in OPTIONAL_FEATURES] + ["use_close", "use_sentiment_lag1"]
optuna.visualization.plot_param_importances(
    study, params=FEATURE_PARAMS
).write_html(f"{BASE_DIR}/param_importance.html")
'''

'\n# ===============================\n# 7️⃣ HYPERPARAMETER OPTIMIZATION\n# ===============================\nimport sqlite3\nfrom optuna.trial import TrialState\n\nprint("\n" + "="*80)\nprint(f"HYPERPARAMETER OPTIMIZATION — {EXPERIMENT_NAME} ({GRN_CLASS_NAME})")\nprint("="*80)\n\nstorage = optuna.storages.RDBStorage(url=STORAGE)\nstudy = optuna.create_study(\n    study_name=STUDY_NAME,\n    direction="minimize",\n    sampler=TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS,\n                       n_ei_candidates=24, multivariate=True),\n    pruner=optuna.pruners.NopPruner(),   # tidak ada pruning — semua trial berjalan penuh\n    storage=storage,\n    load_if_exists=True,\n)\n\n# Bersihkan RUNNING trials dari sesi yang terputus\ndb_path = STORAGE.replace("sqlite:///", "")\nconn    = sqlite3.connect(db_path)\ncur     = conn.cursor()\ncur.execute("SELECT COUNT(*) FROM trials WHERE state = 0")\nn_run   = cur.fetchone()[0]\nif n_run > 0:\n    print(f"🧹 Membersihkan {n_run} RUNNING t

# 8

In [ ]:
# ===============================
# 8️⃣ FINAL MODEL EVALUATION + MODEL DEPLOYMENT EXPORT
# ===============================
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="The 'predict_dataloader' does not have many workers")
warnings.filterwarnings("ignore", message="Attribute 'loss' is an instance")
warnings.filterwarnings("ignore", message="Attribute 'logging_metrics' is an instance")
import logging, json, shutil
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

from lightning.pytorch.callbacks import Callback, ModelCheckpoint

print("\n" + "="*80)
print(f"FINAL EVALUATION — {EXPERIMENT_NAME}")
print(f"Fitur terpilih Optuna ({len(best_config['features'])}): {best_config['features']}")
print("="*80)

class LossHistory(Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses   = []

    def on_train_epoch_end(self, trainer, pl_module):
        tl = trainer.callback_metrics.get("train_loss_epoch")
        if tl is not None:
            self.train_losses.append(float(tl))

    def on_validation_epoch_end(self, trainer, pl_module):
        vl = trainer.callback_metrics.get("val_loss")
        if vl is not None:
            self.val_losses.append(float(vl))


def evaluate_model_on_fold(train_df, val_df, test_df, best_config, fold_num):
    print(f"\n🔄 Fold {fold_num}/{N_FOLDS} [{EXPERIMENT_NAME}]...")

    training = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="close", group_ids=["series_id"],
        max_encoder_length=best_config["encoder_len"],
        max_prediction_length=MAX_PREDICTION_LENGTH,
        time_varying_known_reals=["time_idx"],
        time_varying_unknown_reals=best_config["features"],
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    val_ds = TimeSeriesDataSet.from_dataset(
        training, val_df, predict=True, stop_randomization=True
    )

    nw = 2 if len(train_df) > 500 else 0
    tl = training.to_dataloader(
        train=True, batch_size=best_config["batch"],
        num_workers=nw, pin_memory=torch.cuda.is_available(),
        persistent_workers=(nw > 0)
    )
    vl = val_ds.to_dataloader(
        train=False, batch_size=best_config["batch"],
        num_workers=nw, pin_memory=torch.cuda.is_available(),
        persistent_workers=(nw > 0)
    )

    model = TemporalFusionTransformer.from_dataset(
        training, hidden_size=best_config["hidden"], dropout=best_config["dropout"],
        attention_head_size=best_config["heads"],
        learning_rate=best_config["lr"], loss=RMSE(),
    )
    grn_instances = [m for m in model.modules() if type(m).__name__ == GRN_CLASS_NAME]
    assert len(grn_instances) > 0, f"❌ {GRN_CLASS_NAME} tidak ditemukan dalam model!"
    print(f"   ✅ {len(grn_instances)} instance {GRN_CLASS_NAME} terverifikasi")

    loss_history  = LossHistory()
    checkpoint_cb = ModelCheckpoint(
        dirpath=BASE_DIR,
        filename=f"tmp_fold{fold_num}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    trainer = Trainer(
        max_epochs=100,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        precision="32", gradient_clip_val=best_config["clipnorm"],
        enable_checkpointing=True, logger=False,
        enable_progress_bar=True, enable_model_summary=False,
        profiler=None, deterministic=False, log_every_n_steps=1,
        callbacks=[
            EarlyStopping(monitor="val_loss",
                          patience=EARLY_STOPPING_PATIENCE_FINAL, mode="min"),
            loss_history,
            checkpoint_cb,
        ],
    )
    trainer.fit(model, tl, vl)

    # Load bobot epoch terbaik (val_loss terendah)
    best_model = TemporalFusionTransformer.load_from_checkpoint(
        checkpoint_cb.best_model_path,
        map_location="cpu",
    )
    best_model.eval()

    final_train_loss = trainer.callback_metrics.get("train_loss_epoch")
    final_val_loss   = trainer.callback_metrics.get("val_loss")
    final_train_loss = final_train_loss.item() if final_train_loss is not None else None
    final_val_loss   = final_val_loss.item()   if final_val_loss   is not None else None

    def rolling_predict(base_df, eval_df, n_steps, mdl):
        all_actuals, all_preds = [], []
        full_df = pd.concat([base_df, eval_df], ignore_index=True)
        full_df["time_idx"] = np.arange(len(full_df))
        start_idx = len(base_df)

        for step in range(n_steps):
            pred_idx  = start_idx + step
            enc_start = pred_idx - ENCODER_LENGTH
            window_df = full_df.iloc[enc_start : pred_idx + 1].copy()
            window_df["time_idx"] = np.arange(len(window_df))

            try:
                ds_win = TimeSeriesDataSet(
                    window_df,
                    time_idx="time_idx", target="close", group_ids=["series_id"],
                    max_encoder_length=ENCODER_LENGTH,
                    max_prediction_length=MAX_PREDICTION_LENGTH,
                    time_varying_known_reals=["time_idx"],
                    time_varying_unknown_reals=best_config["features"],
                    add_relative_time_idx=True, add_target_scales=True,
                    add_encoder_length=True,
                )
                dl  = ds_win.to_dataloader(train=False, batch_size=1, num_workers=0)
                out = mdl.predict(dl, mode="raw", return_x=True)
                p   = float(out.output["prediction"].detach().cpu().numpy().flatten()[0])
                a   = float(full_df.iloc[pred_idx]["close"])
                all_preds.append(p)
                all_actuals.append(a)
            except Exception as e:
                print(f"      ⚠️ Step {step + 1} error: {e}")

        return np.array(all_actuals), np.array(all_preds)

    print(f"   Rolling prediction val  ({EVAL_WINDOW} steps)...")
    val_act, val_pred   = rolling_predict(train_df, val_df, EVAL_WINDOW, best_model)

    print(f"   Rolling prediction test ({EVAL_WINDOW} steps)...")
    test_act, test_pred = rolling_predict(
        pd.concat([train_df, val_df], ignore_index=True),
        test_df, EVAL_WINDOW, best_model
    )

    train_close = train_df["close"].values
    val_m  = calculate_metrics(val_act,  val_pred,  train_actuals=train_close)
    test_m = calculate_metrics(test_act, test_pred, train_actuals=train_close)

    print(f"   Val  ({EVAL_WINDOW} pred) — RMSE:{val_m['rmse']:.4f}  "
          f"MAPE:{val_m['mape']:.2f}%  sMAPE:{val_m['smape']:.2f}%  "
          f"MASE:{val_m['mase']:.4f}  DA:{val_m['da']:.1f}%")
    print(f"   Test ({EVAL_WINDOW} pred) — RMSE:{test_m['rmse']:.4f}  "
          f"MAPE:{test_m['mape']:.2f}%  sMAPE:{test_m['smape']:.2f}%  "
          f"MASE:{test_m['mase']:.4f}  DA:{test_m['da']:.1f}%")

    if final_train_loss is not None and final_val_loss is not None:
        gap_pct = ((final_val_loss - final_train_loss) / (final_train_loss + 1e-8)) * 100
        if gap_pct > 50:
            fit_status = "⚠️ Overfitting kuat"
        elif gap_pct > 20:
            fit_status = "⚠️ Indikasi overfitting"
        elif gap_pct < -50:
            fit_status = "ℹ️ Validation loss jauh lebih rendah dari training"
        else:
            fit_status = "✅ Relatif stabil"
        print(f"   Fit  — train_loss={final_train_loss:.4f}  val_loss={final_val_loss:.4f}  "
              f"gap={gap_pct:+.1f}%  {fit_status}")
    else:
        gap_pct    = None
        fit_status = "N/A"

    return {
        "fold":               fold_num,
        "val_predictions":    val_pred,  "val_actuals":  val_act,  "val_metrics":  val_m,
        "test_predictions":   test_pred, "test_actuals": test_act, "test_metrics": test_m,
        "train_size":         len(train_df),
        "val_size":           len(val_df),
        "test_size":          len(test_df),
        "final_train_loss":   final_train_loss,
        "final_val_loss":     final_val_loss,
        "gap_pct":            gap_pct,
        "fit_status":         fit_status,
        "train_loss_history": loss_history.train_losses,
        "val_loss_history":   loss_history.val_losses,
        # untuk deployment & variable importance — tidak perlu training ulang
        "_ckpt_path":         checkpoint_cb.best_model_path,
        "_train_df":          train_df,
        "_val_df":            val_df,
        "_model":             best_model,
        "_val_dl":            vl,
        "_training_dataset":  training,
    }


all_fold_results = []
for i, (tr, vl, te) in enumerate(fold_splits):
    result = evaluate_model_on_fold(tr, vl, te, best_config, i + 1)
    all_fold_results.append(result)


# ===============================
# DEPLOYMENT EXPORT — Fallback Cascade
# ===============================
STABILITY_THRESHOLD = 50.0

print("\n" + "="*80)
print("DEPLOYMENT EXPORT — Pemilihan Fold Stabil (Fallback Cascade)")
print(f"Threshold stabilitas: |gap_pct| ≤ {STABILITY_THRESHOLD}%")
print("="*80)

selected_result = None
for result in reversed(all_fold_results):
    gap      = result["gap_pct"]
    fold_num = result["fold"]
    if gap is None:
        print(f"   Fold {fold_num:2d} — gap: N/A        → ⏭️  Skip")
        continue
    stable = abs(gap) <= STABILITY_THRESHOLD
    print(f"   Fold {fold_num:2d} — gap: {gap:+7.1f}%  → {'✅ LOLOS' if stable else '⏭️  Skip'}")
    if stable and selected_result is None:
        selected_result = result

if selected_result is None:
    selected_result = min(
        all_fold_results,
        key=lambda r: abs(r["gap_pct"]) if r["gap_pct"] is not None else float("inf")
    )
    print(f"\n⚠️  Tidak ada fold dengan |gap| ≤ {STABILITY_THRESHOLD}%.")
    print(f"   Fallback: fold {selected_result['fold']} (|gap| terkecil).")

selected_fold = selected_result["fold"]
print(f"\n✅ Model yang diekspor: Fold {selected_fold}")
print(f"   gap_pct    : {selected_result['gap_pct']:+.1f}%")
print(f"   fit_status : {selected_result['fit_status']}")
print(f"   val  RMSE  : {selected_result['val_metrics']['rmse']:.4f}")
print(f"   test RMSE  : {selected_result['test_metrics']['rmse']:.4f}")

DEPLOY_DIR        = f"{BASE_DIR}/deployment"
MODEL_CKPT_PATH   = f"{DEPLOY_DIR}/model.ckpt"
METADATA_PATH     = f"{DEPLOY_DIR}/model_metadata.json"
HISTORY_DATA_PATH = f"{DEPLOY_DIR}/encoder_history.csv"
os.makedirs(DEPLOY_DIR, exist_ok=True)

shutil.copy(selected_result["_ckpt_path"], MODEL_CKPT_PATH)
print(f"\n📦 Checkpoint → {MODEL_CKPT_PATH}")

encoder_history_df = pd.concat(
    [selected_result["_train_df"], selected_result["_val_df"]],
    ignore_index=True
)
encoder_history_df.tail(ENCODER_LENGTH).to_csv(HISTORY_DATA_PATH, index=False)
print(f"📦 Encoder history ({ENCODER_LENGTH} baris) → {HISTORY_DATA_PATH}")

metadata = {
    "experiment":    EXPERIMENT_NAME,
    "activation":    ACTIVATION_NAME,
    "init":          INIT_NAME,
    "grn_class":     GRN_CLASS_NAME,
    "selected_fold": selected_fold,
    "gap_pct":       round(selected_result["gap_pct"], 2),
    "fit_status":    selected_result["fit_status"],
    "val_metrics":   {k: round(float(v), 4) for k, v in selected_result["val_metrics"].items()},
    "test_metrics":  {k: round(float(v), 4) for k, v in selected_result["test_metrics"].items()},
    "model_config": {
        "hidden_size":           best_config["hidden"],
        "dropout":               best_config["dropout"],
        "attention_head_size":   best_config["heads"],
        "learning_rate":         best_config["lr"],
        "encoder_length":        best_config["encoder_len"],
        "max_prediction_length": MAX_PREDICTION_LENGTH,
    },
    "features": {
        "time_varying_unknown_reals": best_config["features"],
        "time_varying_known_reals":   ["time_idx"],
        "target":                     "close",
        "group_ids":                  ["series_id"],
        "add_relative_time_idx":      True,
        "add_target_scales":          True,
        "add_encoder_length":         True,
    },
    "dashboard_info": {
        "input_columns":        best_config["features"],
        "n_features_selected":  len(best_config["features"]),
        "encoder_history_file": "encoder_history.csv",
        "model_file":           "model.ckpt",
    }
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"📦 Metadata → {METADATA_PATH}")

for result in all_fold_results:
    tmp = result.get("_ckpt_path", "")
    if tmp and os.path.exists(tmp) and tmp != MODEL_CKPT_PATH:
        os.remove(tmp)

shutil.make_archive(f"{BASE_DIR}/deployment_{EXPERIMENT_NAME}", "zip", DEPLOY_DIR)
print(f"📦 Zip → {BASE_DIR}/deployment_{EXPERIMENT_NAME}.zip")
print(f"\n✅ EXPORT SELESAI — Model Fold {selected_fold} · {len(best_config['features'])} fitur terpilih")


FINAL EVALUATION — ELU_UnitNorm
Fitur terpilih Optuna (5): ['close', 'sentiment_lag1', 'WMA_10', 'MACD', 'RSI']

🔄 Fold 1/10 [ELU_UnitNorm]...
   ✅ 15 instance GRNELU_UnitNorm terverifikasi


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: UserWarning:

Checkpoint directory /kaggle/working/ELU_UnitNorm exists and is not empty.



   Rolling prediction val  (21 steps)...


2026-05-21 06:12:52.382481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779343972.684195      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779343972.778642      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779343973.513616      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779343973.513653      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779343973.513663      16 computation_placer.cc:177] computation placer alr

...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: UserWarning:

Checkpoint directory /kaggle/working/ELU_UnitNorm exists and is not empty.



   Rolling prediction val  (21 steps)...
   Rolling prediction test (21 steps)...
   Val  (21 pred) — RMSE:56.4850  MAPE:1.58%  sMAPE:1.59%  MASE:1.2338  DA:58.8%
   Test (21 pred) — RMSE:84.8606  MAPE:2.20%  sMAPE:2.21%  MASE:1.9125  DA:52.9%
   Fit  — train_loss=58.2092  val_loss=18.5754  gap=-68.1%  ℹ️ Validation loss jauh lebih rendah dari training

🔄 Fold 8/10 [ELU_UnitNorm]...
   ✅ 15 instance GRNELU_UnitNorm terverifikasi


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: UserWarning:

Checkpoint directory /kaggle/working/ELU_UnitNorm exists and is not empty.



# 11

In [ ]:
# ===============================
# 1️⃣1️⃣ TFT VARIABLE IMPORTANCE (Metode Bawaan TFT)
# ===============================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import warnings
warnings.filterwarnings("ignore")

RESEARCH_VARS = best_config["features"]

print("\n" + "="*80)
print(f"TFT VARIABLE IMPORTANCE — {EXPERIMENT_NAME}")
print(f"Metode : VSN · interpret_output() — model dari cell 8️⃣ (tanpa training ulang)")
print(f"Scope  : Rata-rata dari semua {N_FOLDS} fold walk-forward")
print(f"Variabel: {len(RESEARCH_VARS)} terpilih Optuna → {RESEARCH_VARS}")
print("="*80)

fold_importances = []

for result in all_fold_results:
    fold_num = result["fold"]
    model_vi = result["_model"]
    vl_vi    = result["_val_dl"]

    print(f"  📊 Fold {fold_num:2d}/{N_FOLDS} — menghitung importance...", end=" ", flush=True)

    model_vi.eval()
    with torch.no_grad():
        raw_preds = model_vi.predict(vl_vi, mode="raw", return_x=True)
        interp    = model_vi.interpret_output(raw_preds.output, reduction="sum")

    enc_imp = interp["encoder_variables"].detach().cpu()

    # ── PERBAIKAN: gunakan time_varying_reals_encoder dari hparams ────────────
    # Inilah urutan yang persis sesuai dengan tensor encoder_variables
    # (bukan ds_vi.reals yang mencakup semua termasuk meta internal)
    vsn_vars = list(model_vi.hparams.time_varying_reals_encoder)

    # Map tensor ke nama variabel
    full_map = {name: float(enc_imp[i])
                for i, name in enumerate(vsn_vars)
                if i < len(enc_imp)}

    # Filter hanya variabel terpilih Optuna
    research_map = {v: full_map.get(v, 0.0) for v in RESEARCH_VARS}

    # Renormalisasi agar total = 1.0
    total = sum(research_map.values())
    if total > 0:
        research_map = {k: v / total for k, v in research_map.items()}

    fold_importances.append(research_map)

    raw_str = " | ".join([f"{v}:{full_map.get(v,0):.3f}" for v in RESEARCH_VARS])
    print(f"✅  raw→ {raw_str}")

# ── Rata-rata & std lintas fold ───────────────────────────────────────────────
mean_imp = {v: float(np.mean([fi[v] for fi in fold_importances])) for v in RESEARCH_VARS}
std_imp  = {v: float(np.std ([fi[v] for fi in fold_importances])) for v in RESEARCH_VARS}
mean_imp_sorted = dict(sorted(mean_imp.items(), key=lambda x: x[1], reverse=True))

print("\n" + "="*80)
print(f"📋 VARIABLE IMPORTANCE — Mean ± Std ({len(RESEARCH_VARS)} variabel terpilih Optuna)")
print(f"{'Variabel':<20} {'Mean':>8} {'Std':>8} {'Proporsi':>10}")
print("-" * 52)
for var, imp in mean_imp_sorted.items():
    print(f"  {var:<18} {imp:>8.4f} {std_imp[var]:>8.4f} {imp*100:>9.2f}%")
print(f"  {'TOTAL':<18} {sum(mean_imp_sorted.values()):>8.4f} {'':>8} {'100.00%':>10}")

print(f"\n📋 VARIABLE IMPORTANCE PER FOLD:")
header = f"{'Variabel':<20}" + "".join([f" Fold{i+1:02d}" for i in range(N_FOLDS)])
print(header)
print("-" * (20 + 7 * N_FOLDS))
for var in mean_imp_sorted:
    row = f"  {var:<18}"
    for fi in fold_importances:
        row += f" {fi[var]*100:5.1f}%"
    print(row)

# ── Bar chart ─────────────────────────────────────────────────────────────────
vars_list  = list(mean_imp_sorted.keys())
means_pct  = [mean_imp_sorted[v] * 100 for v in vars_list]

def get_color(name):
    if name == "close":               return "#2196F3"
    elif name in OPTIONAL_FEATURES:   return "#4CAF50"
    elif "sentiment" in name.lower(): return "#FF9800"
    else:                             return "#9E9E9E"

colors     = [get_color(v) for v in vars_list]
fig_height = max(5, len(vars_list) * 0.7 + 2)

fig_vi, ax_vi = plt.subplots(figsize=(14, fig_height))
bars = ax_vi.barh(
    list(range(len(vars_list))),
    means_pct[::-1],
    color=colors[::-1],
    edgecolor="white", linewidth=0.5,
)
ax_vi.set_yticks(list(range(len(vars_list))))
ax_vi.set_yticklabels(vars_list[::-1], fontsize=11)
for bar, m in zip(bars, means_pct[::-1]):
    ax_vi.text(bar.get_width() + 0.3,
               bar.get_y() + bar.get_height() / 2,
               f"{m:.2f}%", va="center", ha="left", fontsize=10)
ax_vi.set_xlabel("Variable Importance — Mean across folds (%)", fontsize=12)
ax_vi.set_title(
    f"TFT Variable Importance — {EXPERIMENT_NAME}\n"
    f"VSN Encoder · {N_FOLDS} Fold Walk-Forward · {len(RESEARCH_VARS)} Variabel Terpilih Optuna",
    fontsize=13, fontweight="bold"
)
ax_vi.set_xlim(0, max(means_pct) * 1.2)
ax_vi.grid(axis="x", alpha=0.3)

legend_elements = []
if any(v == "close" for v in RESEARCH_VARS):
    legend_elements.append(mpatches.Patch(facecolor="#2196F3", label="Close (harga)"))
if any(v in OPTIONAL_FEATURES for v in RESEARCH_VARS):
    legend_elements.append(mpatches.Patch(facecolor="#4CAF50", label="Teknikal Indikator"))
if any("sentiment" in v.lower() for v in RESEARCH_VARS):
    legend_elements.append(mpatches.Patch(facecolor="#FF9800", label="Sentimen"))
ax_vi.legend(handles=legend_elements, loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/variable_importance_{EXPERIMENT_NAME}.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"\n✅ Bar chart → {BASE_DIR}/variable_importance_{EXPERIMENT_NAME}.png")

# ── Pie chart ─────────────────────────────────────────────────────────────────
groups = {}
close_imp = mean_imp.get("close", 0)
if close_imp > 1e-6:
    groups["Close (harga)"] = close_imp
tech_imp = sum(mean_imp.get(f, 0) for f in OPTIONAL_FEATURES if f in RESEARCH_VARS)
if tech_imp > 1e-6:
    groups["Teknikal Indikator"] = tech_imp
sent_imp = sum(v for k, v in mean_imp.items() if "sentiment" in k.lower())
if sent_imp > 1e-6:
    groups["Sentimen"] = sent_imp

pie_color_map = {
    "Close (harga)":      "#2196F3",
    "Teknikal Indikator": "#4CAF50",
    "Sentimen":           "#FF9800",
}
pie_colors = [pie_color_map[k] for k in groups]

fig_pie, ax_pie = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax_pie.pie(
    groups.values(), labels=groups.keys(), autopct="%1.1f%%",
    colors=pie_colors, startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=2)
)
for t in autotexts:
    t.set_fontsize(12); t.set_fontweight("bold")
ax_pie.set_title(
    f"Kontribusi Grup Fitur — {EXPERIMENT_NAME}\n"
    f"VSN Encoder · {N_FOLDS} Fold · {len(RESEARCH_VARS)} Variabel Terpilih Optuna",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/variable_importance_pie_{EXPERIMENT_NAME}.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Pie chart → {BASE_DIR}/variable_importance_pie_{EXPERIMENT_NAME}.png")

# ── Heatmap variabel × fold ───────────────────────────────────────────────────
heatmap_data = np.array([
    [fold_importances[f][v] * 100 for f in range(N_FOLDS)]
    for v in vars_list
])
fig_hm, ax_hm = plt.subplots(figsize=(14, max(4, len(vars_list) * 0.7 + 2)))
im = ax_hm.imshow(heatmap_data, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax_hm, label="Importance (%)")
ax_hm.set_xticks(range(N_FOLDS))
ax_hm.set_xticklabels([f"Fold {i+1}" for i in range(N_FOLDS)], fontsize=9)
ax_hm.set_yticks(range(len(vars_list)))
ax_hm.set_yticklabels(vars_list, fontsize=10)
for i in range(len(vars_list)):
    for j in range(N_FOLDS):
        val = heatmap_data[i, j]
        ax_hm.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                   color="white" if val > heatmap_data.max() * 0.6 else "black")
ax_hm.set_title(
    f"Variable Importance per Fold — {EXPERIMENT_NAME}\n"
    f"Setiap sel = importance (%) · {len(RESEARCH_VARS)} Variabel Terpilih Optuna",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/variable_importance_heatmap_{EXPERIMENT_NAME}.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Heatmap → {BASE_DIR}/variable_importance_heatmap_{EXPERIMENT_NAME}.png")

# ── Simpan JSON ───────────────────────────────────────────────────────────────
import json as _json
vi_results = {
    "experiment":          EXPERIMENT_NAME,
    "method":              "TFT VSN interpret_output — model reused from cell 8 (no retraining)",
    "vsn_source":          "time_varying_reals_encoder from model.hparams",
    "scope":               f"Mean across all {N_FOLDS} walk-forward folds",
    "n_features_selected": len(RESEARCH_VARS),
    "features_selected":   RESEARCH_VARS,
    "note":                "Renormalized to Optuna-selected variables only. Internal TFT features discarded.",
    "mean_importance":     {k: round(float(v), 6) for k, v in mean_imp_sorted.items()},
    "std_importance":      {k: round(float(std_imp[k]), 6) for k in mean_imp_sorted},
    "group_importance":    {k: round(float(v), 6) for k, v in groups.items()},
    "per_fold_importance": [{k: round(float(v), 6) for k, v in fi.items()}
                            for fi in fold_importances],
}
with open(f"{BASE_DIR}/variable_importance_{EXPERIMENT_NAME}.json", "w") as f:
    _json.dump(vi_results, f, indent=2)
print(f"✅ JSON → {BASE_DIR}/variable_importance_{EXPERIMENT_NAME}.json")

print("\n" + "="*80)
print("📝 CATATAN METODOLOGI:")
print(f"   • {len(RESEARCH_VARS)} variabel dipilih Optuna dari kandidat awal.")
print("   • Urutan VSN diambil dari model.hparams.time_varying_reals_encoder")
print("     — inilah urutan tensor encoder_variables yang sebenarnya.")
print("   • Fitur meta TFT (time_idx, relative_time_idx) masuk tensor tapi")
print("     bukan variabel penelitian → dibuang saat filter RESEARCH_VARS.")
print("   • Renormalisasi terhadap variabel penelitian saja → total = 100%.")
print("   • Mean ± Std dari 10 fold = estimasi ekspektasi lintas waktu.")
print("="*80)

In [ ]:
import optuna
import pandas as pd

study = optuna.load_study(
    study_name="tft_walkforward_10fold_ELU_UnitNorm",
    storage=STORAGE
)

df = study.trials_dataframe(
    attrs=("number", "state", "value", "params")
)

display(df)

import matplotlib.pyplot as plt

# ambil dataframe dari optuna study
trials_df = study.trials_dataframe(attrs=("number", "state", "value", "params"))

# filter hanya trial yang COMPLETE
complete_trials = trials_df[trials_df["state"] == "COMPLETE"]

# plot
plt.figure(figsize=(10,5))

plt.plot(
    complete_trials["number"],
    complete_trials["value"],
    marker="o",
    linestyle="-",
    label="Trial Value"
)

best_idx = complete_trials["value"].idxmin()  # gunakan idxmax() jika maximize
best_trial = complete_trials.loc[best_idx]

plt.scatter(
    best_trial["number"],
    best_trial["value"],
    color="red",
    s=120,
    label="Best Trial"
)

plt.xlabel("Trial Number")
plt.ylabel("Objective Value")
plt.title("Optuna Trials Performance (Complete Only)")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

# 9

In [ ]:
# ===============================
# 9️⃣ AGGREGATE & SAVE
# ===============================
def agg_metric(results, split, metric, fn=np.mean):
    vals = [r[f"{split}_metrics"][metric] for r in results]
    vals = [v for v in vals if not np.isnan(v)]   # skip NaN (misal fold hanya 1 titik)
    return float(fn(vals)) if vals else float("nan")

METRICS_LIST = ["rmse", "mape", "smape", "mase", "da"]

val_agg  = {f"{m}_{s}": agg_metric(all_fold_results, "val",  m, fn)
            for m in METRICS_LIST
            for s, fn in [("mean", np.mean), ("std", np.std)]}
test_agg = {f"{m}_{s}": agg_metric(all_fold_results, "test", m, fn)
            for m in METRICS_LIST
            for s, fn in [("mean", np.mean), ("std", np.std)]}

print(f"\n{'='*80}")
print(f"HASIL AKHIR — {EXPERIMENT_NAME}")
print(f"{'='*80}")
print(f"\n📊 VALIDATION ({N_FOLDS} folds × {EVAL_WINDOW} prediksi):")
for m in METRICS_LIST:
    print(f"   {m.upper():6s}: {val_agg[m+'_mean']:.4f} ± {val_agg[m+'_std']:.4f}")
print(f"\n📊 TEST ({N_FOLDS} folds × {EVAL_WINDOW} prediksi):")
for m in METRICS_LIST:
    print(f"   {m.upper():6s}: {test_agg[m+'_mean']:.4f} ± {test_agg[m+'_std']:.4f}")

# Ringkasan fit status per fold
print(f"\n📊 FIT STATUS PER FOLD:")
for r in all_fold_results:
    g = f"{r['gap_pct']:+.1f}%" if r["gap_pct"] is not None else "N/A"
    print(f"   Fold {r['fold']:2d} | train={r['final_train_loss'] or 'N/A':.4f}  "
          f"val={r['final_val_loss'] or 'N/A':.4f}  gap={g}")

def to_native(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, dict):        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):        return [to_native(i) for i in obj]
    return obj

results_summary = to_native({
    "experiment":    EXPERIMENT_NAME,
    "activation":    ACTIVATION_NAME,
    "init":          INIT_NAME,
    "grn_class":     GRN_CLASS_NAME,
    "methodology":   f"Walk-Forward {N_FOLDS} folds × {EVAL_WINDOW} prediksi 1-step",
    "n_folds":       N_FOLDS,
    "eval_window":   EVAL_WINDOW,
    "encoder_length": ENCODER_LENGTH,
    "initial_train_size": initial_train_size,
    "val_size_per_fold":  val_size,
    "test_size_per_fold": test_size,
    "best_config":        best_config,
    "validation_aggregate_metrics": val_agg,
    "test_aggregate_metrics":       test_agg,
    "fold_results": [
        {"fold":       r["fold"],
         "validation": {k: float(v) for k, v in r["val_metrics"].items()},
         "test":       {k: float(v) for k, v in r["test_metrics"].items()},
         "sizes":      {"train": r["train_size"], "val": r["val_size"], "test": r["test_size"]},
         "fit":        {"train_loss": r["final_train_loss"],
                        "val_loss":   r["final_val_loss"],
                        "gap_pct":    r["gap_pct"],
                        "status":     r["fit_status"]}}
        for r in all_fold_results
    ]
})
with open(f"{BASE_DIR}/{RESULTS_FILENAME}", "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"\n💾 Tersimpan: {BASE_DIR}/{RESULTS_FILENAME}")

# 10

In [ ]:
# ===============================
# 🔟 VISUALIZATION
# ===============================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

folds = [r["fold"] for r in all_fold_results]

# ── Grafik 1: Metrik per fold — 1 baris × 5 kolom ────────────────────────────
fig1, axes1 = plt.subplots(1, 5, figsize=(30, 5))
fig1.suptitle(
    f"TFT {ACTIVATION_NAME} + Init {INIT_NAME} — "
    f"Walk-Forward {N_FOLDS} Folds × {EVAL_WINDOW} Prediksi",
    fontsize=15, fontweight="bold"
)
metrics_plot = [
    ("rmse",  "RMSE"),
    ("mape",  "MAPE (%)"),
    ("smape", "sMAPE (%)"),
    ("mase",  "MASE"),
    ("da",    "Directional Accuracy (%)"),
]
for idx, (metric, label) in enumerate(metrics_plot):
    ax = axes1[idx]
    v  = [r["val_metrics"][metric]  for r in all_fold_results]
    t  = [r["test_metrics"][metric] for r in all_fold_results]
    ax.plot(folds, v, marker="o", lw=2, ms=8, label="Validation", color="blue")
    ax.plot(folds, t, marker="s", lw=2, ms=8, label="Test",       color="red")
    v_mean = np.nanmean(v)
    t_mean = np.nanmean(t)
    ax.axhline(y=v_mean, color="blue", ls="--", alpha=0.5,
               label=f"Val mean={v_mean:.2f}")
    ax.axhline(y=t_mean, color="red",  ls="--", alpha=0.5,
               label=f"Test mean={t_mean:.2f}")
    if metric == "mase":
        ax.axhline(y=1.0, color="green", ls=":", lw=1.5, alpha=0.8,
                   label="MASE=1 (setara naive)")
    if metric == "da":
        ax.axhline(y=50.0, color="gray", ls=":", lw=1.5, alpha=0.8,
                   label="DA=50% (random)")
    ax.set_xlabel("Fold")
    ax.set_ylabel(label)
    ax.set_title(f"{label} — {EXPERIMENT_NAME}", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/walkforward_metrics_{EXPERIMENT_NAME}.png",
            dpi=300, bbox_inches="tight")
plt.show()

# ── Grafik 2: Learning Curve per fold — 2 baris × 5 kolom ───────────────────
n_cols = 5
n_rows = 2

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(30, 12))
fig2.suptitle(
    f"Learning Curve per Fold — {EXPERIMENT_NAME}\n"
    f"Train Loss vs Validation Loss per Epoch",
    fontsize=14, fontweight="bold"
)

for idx, r in enumerate(all_fold_results):
    ax     = axes2[idx // n_cols][idx % n_cols]
    t_hist = r["train_loss_history"]
    v_hist = r["val_loss_history"]
    epochs = list(range(1, len(t_hist) + 1))

    ax.plot(epochs, t_hist, color="red",  lw=2, label="Training")
    ax.plot(epochs, v_hist, color="blue", lw=2, label="Validation")

    # Warna background sesuai status
    status = r["fit_status"]
    if "OVERFIT" in status:
        bg_color     = "#fff0f0"
        status_label = "Overfitting"
    elif "UNDERFIT" in status:
        bg_color     = "#fff8e1"
        status_label = "Underfitting"
    elif "NORMAL" in status or "FIT BAIK" in status:
        bg_color     = "#f0fff0"
        status_label = "Good Fit"
    else:
        bg_color     = "#f5f5f5"
        status_label = "N/A"

    ax.set_facecolor(bg_color)

    gp = f"{r['gap_pct']:+.1f}%" if r["gap_pct"] is not None else "N/A"
    ax.set_title(
        f"Fold {r['fold']} — (gap={gp})",
        fontweight="bold", fontsize=10
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (normalized scale)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# N_FOLDS = 10, n_cols = 5, n_rows = 2 → pas, tidak ada subplot kosong
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/learning_curve_{EXPERIMENT_NAME}.png",
            dpi=300, bbox_inches="tight")
plt.show()
import shutil

shutil.make_archive('/kaggle/working/ELU_UnitNorm', 'zip', '/kaggle/working/ELU_UnitNorm')
print(f"\n✅ SELESAI — {EXPERIMENT_NAME}")